# 🫀 퀘스트 46 · Q7-F — **P 창 심박수 교란: 형태인가, 밀린 창인가**

| | **MedKOS / `notebooks/quest46_q7f_window_confound.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0056`(Q7-H) · `ailab-2026-0055`(Q7-E) · `ailab-2026-0054`(Q7-D) |
| 규약 | **R11-c · R17 · R20 · R21 · R22 · R23** |
| 학습 | **0회** — `svdb_data5.npz` 의 파형만 읽는다 |

## 이 실험은 **새 발견이 아니라 정오 확인**이다

Q7-D·E·H 는 전부 「형태 축(P영역)」을 파고들었다. 그런데 그 P영역은 **R 기준 고정창**이다.

```
비트 배열 300샘플 @360Hz · R = index 100  (_RPRE)
P_SEG = (0, 85)  →  R−278ms ~ R−42ms
```

심박수가 다르면 **창 안에 든 것이 다르다.** `#865` 로 어림하면:

| | RR | 직전 R | 직전 T 대략 | P 창(−278~−42ms) 안? |
|---|---:|---:|---:|---|
| **S** (142 bpm) | 423 ms | −423 ms | **−223 ms** | **들어온다** |
| **N** (77 bpm) | 779 ms | −779 ms | −499 ms | 안 들어온다 |

*(Bazett QT 로 잡은 어림 — 【F-A】에서 실측 대체한다.)*

그러면 두 템플릿 점수가 잡는 게 **P 파 형태가 아니라 「창 안에 T 가 있냐 없냐」** 일 수
있다. 그리고 Q7-H 결과가 정확히 그 모양으로 의심스럽다:

- 기여 상위가 `#879`(유병률 **0.023**) · `#800`(**0.016**) — **고칠 기준 오염이 없는**
  저부담 개체에서 Δ +0.33
- 두 템플릿 매크로 **0.9008** 이 RR 위치형 **0.9367** 바로 밑에 붙는다
- 두 템플릿은 이런 인공물을 **가장 효율적으로** 빨아먹는 형태다 — S 비트들이 공유하는
  「창이 밀린 정도」를 `medS` 가 그대로 담는다

**이게 사실이면 Q7-D·E·H 의 형태 축 해석이 통째로 리듬 이야기다.** 이미 기록에 들어간
결론의 정오 문제라 새 실험보다 먼저 간다.

## 창을 셋으로 쪼갠다 — 위치로 답이 갈린다

직전 T 는 **직전 R 이후 0.15~0.55 RR** 구간에 있다. 현재 R 기준 좌표로는
`[100 − 0.85·pre, 100 − 0.45·pre]` (샘플 단위). 대입해 보면:

| pre_rr | 심박수 | 직전 T 가 앉는 index | 비고 |
|---:|---:|---|---|
| 150 | 144 bpm | **−28 ~ 32** | **창 앞부분을 덮는다** |
| 200 | 108 bpm | −70 ~ 10 | 살짝 걸친다 |
| 280 | 77 bpm | −138 ~ −26 | 창 밖 |

반면 **P 파**는 PR 120~200ms · 폭 80~110ms 라 대략 **index 30~78**, 봉우리는 60 근처다.
→ 두 신호가 **다른 자리에 앉는다.** 그래서 창을 쪼개면 갈린다:

| 구간 | index | 시간 | 무엇이 있나 |
|---|---|---|---|
| `P_full` | 0–85 | −278 ~ −42ms | Q7-H 가 쓴 창 (재현용) |
| **`P_early`** | **0–32** | −278 ~ −189ms | **빠른 비트에서 직전 T 가 드는 자리** |
| **`P_late`** | **53–85** | −131 ~ −42ms | **P 파 봉우리·끝이 사는 자리** |
| `STT` | 130–215 | +83 ~ +319ms | **음성 대조** (P 와 같은 폭 85) |

`P_early` 와 `P_late` 는 **폭이 같다(32)**. 가운데 33–52 는 애매 구역이라 일부러 버린다.

## 사전등록

| 관문 | 내용 | 문턱 | **지지의 의미** |
|---|---|---|---|
| **F1** | **심박수 정합** — 같은 `pre_rr` 대역 안에서만 S/N 쌍을 세도 유지되나 | 짝지은 (정합 − 무정합) CI 하한 **> −0.05** | 형태가 진짜 |
| **F2** | **신호의 자리** — `P_late` 가 `P_early` 보다 나은가 | 짝지은 (late − early) CI 하한 **> 0** | 형태가 진짜 |
| **F3** | **음성 대조** — `P_full` 이 `STT` 보다 의미 있게 나은가 | 짝지은 (P_full − STT) CI 하한 **> 0.05** | 신호가 P **창** 안에 있다 |
| **F4** | ⚠️ **교란 확인** — 직전 T 중첩 격차가 클수록 `P_early` 가 잘 맞히나 | `rho(중첩격차, P_early)` CI 하한 **> 0** | **교란이 있다** |

**F4 만 지지의 방향이 반대다** — F4 지지 = 나쁜 소식이다. 헷갈리지 않게 표에 박아둔다.

⚠️ **F3 이 말할 수 있는 것의 한계**: F3 지지는 「신호가 P **창** 안에 있다」까지다.
**「그 신호가 P 파다」는 뜻이 아니다** — 직전 T 가 창에 들어와도 F3 은 통과한다.
실제로 픽스처의 T 침입 시나리오(P 파가 S·N 완전히 동일)에서 **F3 이 +0.36 으로 지지**됐다.
P 인지 T 인지는 **F2 가 가른다**(창 안 어느 자리냐).

### 판정 조합

- **F1 ✅ · F2 ✅ · F3 ✅ · F4 ❌** → 형태 축은 진짜다. Q7-D·E·H 해석 유지, Q7-G 진행
- **F1 ❌ 또는 F2 ❌, F4 ✅** → **리듬 대리다.** Q7-D·E·H 형태 축 해석을 **소급 정정**하고
  Q7-G 취소. 큐는 Q7-I(리듬 특징 묶음)로 직행
- 갈리면 **갈렸다는 사실을 결과로 쓴다**(R18) — 억지로 한쪽으로 읽지 않는다

### ⚠️ 이 설계가 **판단하지 못하는** 리듬 — 먼저 밝혀둔다

지금까지의 모든 실험(Q7-B~H)은 **암묵적으로 세 가지를 가정**했다.
① 안정된 기저 리듬(N)이 있고 ② S 는 그것보다 **이르며** ③ 각 비트에 **불연속 P 파**가
R 앞 고정창에 하나 있다. 아래 리듬들은 이 가정을 **각각 다른 방식으로** 깬다.

| 리듬 | 무엇이 깨지나 | 지금 설계의 반응 |
|---|---|---|
| **심방세동(AFIB)** | ②③ 둘 다. 기저 RR 이 불규칙해 「평소」가 없고, P 대신 f 파가 **연속**으로 깔린다 | RR 위치형은 잡음이 된다. 템플릿 평균은 위상 고정이 안 된 f 파를 **뭉개 없앤다** → 거리 특징이 무의미 |
| **심방조동(AFL)** | ③. F 파가 ~300/min 으로 **계속** 있다. 고정 전도비가 아니면 비트마다 **F 파 위상이 다르다** | 창 안 내용물이 비트마다 달라진다 — **Q7-F 가 쫓는 것과 같은 종류의 인공물** |
| **SVT·심방빈맥(지속 런)** | ②. 런의 **첫 비트만** 이르다. 2번째부터는 직전 비트 대비 이르지 않다 | 레코드 중앙 RR 기준이라 런 전체가 「이르다」로 찍힌다. 런이 길면 **중앙값 자체가 끌려간다** — `#865`(S 57.6% · 142bpm 규칙적)가 정확히 이 모양이다 |
| **접합부 조기박동(`J`)** | ③. 역행성 P 가 **QRS 안이나 직후**에 있다 | AAMI 에서 `J` 는 **S 클래스**다. 그 비트의 신호는 P 창이 아니라 **`STT` 구간**에 있다 → **F3 의 음성 대조 가정이 깨진다** |

**라벨 쪽 문제도 같이 있다.** WFDB 규약상 AFIB/AFL 은 **리듬 주석**(`(AFIB`)이고
그 안의 전도된 비트는 대개 **`N` 으로 표기**된다 — 즉 AAMI 3-class 에서 **AF 는 N 이다**.
그래서 이 데이터셋으로는 「AF 를 맞힌다/틀린다」를 물을 수조차 없다.
⚠️ 이건 규약에 근거한 **예상**이다 — 【F-E】에서 **실측으로 확인**한다.

**그런데 우리는 이걸 볼 수 있는 재료를 이미 갖고 있었다.** `svdb_data5.npz` 에
`rhythm`·`rhythm_names`·`sym` 이 들어 있는데 **Q7-D 부터 지금까지 한 번도 안 읽었다.**
【F-E】가 그걸 읽어 층화한다 — **관문이 아니라 진단**이다.

### 설계에서 미리 못 박는 것

- **F1 은 평가만 제한한다.** 템플릿을 정합 부분집합에서 다시 적합하지 않는다 —
  한 번에 하나만 바꿔야 원인이 읽힌다.
- **정합은 검정력을 잃는다.** S 는 정의상 빠르니 `pre_rr` 분포가 겹치는 구간이 좁다.
  **남은 S 비트 수·쌍 수·개체 수를 관문보다 먼저 출력**하고, 부족하면 **미결**이다
  (조용히 적은 n 으로 판정하지 않는다 — R17).
- **두 템플릿 점수는 Q7-H 와 같은 방식으로 교차적합**한다(R22). 인-샘플이면 신호 0에서도
  0.89 가 나온다 — 그 함정은 이미 겪었다.
- **직전 T 위치는 모형 어림이다**(0.15~0.55 RR). 실측 T 주석이 없으니 그렇게 쓰고,
  **어림이라는 사실을 값 옆에 붙여 출력**한다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0    = 20260803
IDX_S    = 1
RPRE     = 100       # svdb_labels._RPRE — 비트 배열에서 R 위치
NB_BOOT  = 4000      # 레코드 수준 부트스트랩
NB_REC   = 300       # 레코드 내부 부트스트랩(개체별 CI)

# ── 구간. **직전 T 와 P 파가 다른 자리에 앉는다는 게 이 실험의 전부다.**
SEGS = {
    "P_full":  (0, 85),     # Q7-H 가 쓴 창 (재현용)          R−278 ~ −42ms
    "P_early": (0, 32),     # 빠른 비트에서 직전 T 가 드는 자리  R−278 ~ −189ms
    "P_late":  (53, 85),    # P 봉우리·끝이 사는 자리          R−131 ~ −42ms
    "STT":     (130, 215),  # 음성 대조 (P_full 과 같은 폭 85)  R+83 ~ +319ms
}
# 직전 T 는 직전 R 이후 0.15~0.55 RR — 현재 R 기준 [RPRE−0.85·pre, RPRE−0.45·pre]
T_LO, T_HI = 0.85, 0.45     # ★ 모형 어림. 실측 T 주석이 없다 — 값 옆에 그렇게 표기한다

# ── 사전등록 상수. **이 아래 어느 셀에서도 다시 고르지 않는다.**
MIN_S_TPL  = 20      # Q7-H 승계 — 여기서 다시 고르지 않는다
K_FOLD     = 5       # Q7-H 승계 (R22 — 인-샘플이면 신호 0에서도 0.89)
N_REPEAT   = 3       # Q7-H 승계 (겹 배정 잡음을 SE 에 넣는다)
BAND_FRAC  = 0.05    # 심박수 정합 대역폭 = BAND_FRAC × 레코드 중앙 RR
BAND_MIN   = 8       # 대역폭 하한(샘플)
MIN_BAND_S = 3       # 한 대역이 쓸모 있으려면 필요한 S · N 최소 개수
MIN_BAND_N = 3
MIN_MATCH_S = 20     # 정합 후 남은 S 가 이보다 적으면 그 개체는 **미결**
MIN_MATCH_PAIR = 200 # 정합 쌍이 이보다 적으면 미결
MIN_MATCH_REC = 15   # 정합 가능한 개체가 이보다 적으면 F1 자체가 미결
NI_MATCH   = 0.05    # F1 비열등 여유
F3_MARGIN  = 0.05    # F3 — P 가 ST/T 보다 이만큼은 나아야 P 특이성이다

CONFIG = dict(
    exp="quest46_q7f_window_confound", quest="ailab-2026-0046", step="svdb-window-confound",
    parent_exp=["quest46_q7h_two_template", "ailab-2026-0056"],
    purpose=("Q7-D·E·H 의 「형태 축」이 P 파 형태인가, 심박수 때문에 창 안 내용물이 바뀐 "
             "인공물인가를 가른다. 새 발견이 아니라 **이미 기록된 결론의 정오 확인**이다"),
    dataset="SVDB 전수 · Q7-B 예측 캐시(라벨·매핑용) + svdb_data5.npz (학습 0회)",
    segments={k: list(v) for k, v in SEGS.items()},
    t_window_model=[T_LO, T_HI],
    predictions={
        "F1": f"짝지은 (심박수 정합 − 무정합) CI 하한 > −{NI_MATCH}  [지지 = 형태가 진짜]",
        "F2": "짝지은 (P_late − P_early) CI 하한 > 0  [지지 = 형태가 진짜]",
        "F3": (f"짝지은 (P_full − STT) CI 하한 > {F3_MARGIN}  [지지 = 신호가 P **창** 안에 "
               "있다. **P 파라는 뜻은 아니다** — 직전 T 도 창에 들어오면 통과한다]"),
        "F4": "rho(직전T 중첩격차, P_early AUROC) CI 하한 > 0  ⚠️ [지지 = **교란이 있다**]"},
    caveat=("**F4 만 지지의 방향이 반대다** — 지지 = 나쁜 소식. 직전 T 위치는 0.15~0.55 RR "
            "**모형 어림**이며 실측 T 주석이 아니다. F1 은 **평가만** 제한하고 템플릿을 "
            "다시 적합하지 않는다(한 번에 하나만 바꾼다). 정합은 검정력을 잃으므로 "
            "남은 S·쌍·개체 수를 관문보다 먼저 출력하고 부족하면 미결이다(R17). "
            "두 템플릿은 라벨을 쓰는 **상한**이고 매크로를 학습 모델 0.8842 와 비교하지 않는다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7f_window_confound", CONFIG, project=PROJECT)
run.log("설정 ✅ 학습 0회 · 구간 4종 · F4 는 **지지 = 교란 있음**")

In [ ]:
# CELL 2 — 【G0】 자산 · 매핑 (Q7-D/E/H 와 동일 규약 — fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
CNTJ = os.path.join(PROJECT, "data", "svdb_ann_counts.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (CNTJ, "Q7-A 주석 카운트"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
CNT = {int(k): {int(kk): vv for kk, vv in v.items()} for k, v in json.load(open(CNTJ)).items()}
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")

labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

CNT3 = {r: (v.get(0, 0), v.get(1, 0), v.get(2, 0)) for r, v in CNT.items()}
mism = []
for r in np.unique(REC):
    o = tuple(int((Y[REC == r] == k).sum()) for k in range(3)); a = CNT3.get(int(r))
    if not a or any(abs(x - y) > 2 for x, y in zip(o, a)):
        mism.append((int(r), o, a))
run.log("\n" + "=" * 100)
run.log("【G0】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(np.unique(REC))}개 · (N,S,V) 불일치 {len(mism)}건")
for r, o, a in mism:
    run.log(f"    #{r}  예측 {o}  vs 주석 {a}   차 {tuple(x-y for x, y in zip(o, a))}")
CONFIG["mismatch"] = [{"rec": r, "obs": list(o), "ann": list(a) if a else None} for r, o, a in mism]
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【F-A】 적재 + **직전 T 중첩률** (모형 어림)
d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"svdb_data5 와 예측 캐시 길이 불일치 {int(keep.sum())} vs {len(Y)}"
BEAT = np.asarray(d5["beat"])[keep].astype(np.float32)
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【F-A】 코호트 · 창 · 직전 T 중첩")
run.log("=" * 100)
run.log(f"  비트 {BEAT.shape} (R = index {RPRE}) · RR {PRE.shape} · 레코드 {len(ALLR)}개")
for k_, (a_, b_) in SEGS.items():
    run.log(f"    {k_:<8} index {a_:>3}–{b_:<3} (폭 {b_-a_:>3})"
            f"  =  R{(a_-RPRE)/360*1000:+.0f}ms ~ R{(b_-RPRE)/360*1000:+.0f}ms")

def t_overlap(pre_v, seg):
    """직전 T 가 구간 seg 를 덮는 비율. ★ 0.15~0.55 RR **모형 어림**이다 (T 주석 없음)."""
    s0, s1 = seg
    lo = RPRE - T_LO * pre_v          # 직전 T 시작 (현재 R 기준 index)
    hi = RPRE - T_HI * pre_v          # 직전 T 끝
    ov = np.clip(np.minimum(hi, s1) - np.maximum(lo, s0), 0, None)
    return ov / max(s1 - s0, 1)

_ov_e = t_overlap(PRE, SEGS["P_early"]); _ov_l = t_overlap(PRE, SEGS["P_late"])
_tt = (Y == IDX_S)
run.log(f"\n  직전 T 중첩률 (모형 어림 · 전체 비트)")
run.log(f"    P_early — S {_ov_e[_tt].mean():.3f} · N {_ov_e[~_tt].mean():.3f}"
        f"  **격차 {_ov_e[_tt].mean()-_ov_e[~_tt].mean():+.3f}**")
run.log(f"    P_late  — S {_ov_l[_tt].mean():.3f} · N {_ov_l[~_tt].mean():.3f}"
        f"  격차 {_ov_l[_tt].mean()-_ov_l[~_tt].mean():+.3f}")
run.log("    → P_early 격차가 크고 P_late 격차가 0 에 가까우면 **창을 쪼갠 게 유효**하다")
CONFIG["t_overlap_global"] = dict(
    early_s=float(_ov_e[_tt].mean()), early_n=float(_ov_e[~_tt].mean()),
    late_s=float(_ov_l[_tt].mean()), late_n=float(_ov_l[~_tt].mean()))
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【F-B】 ★ 구간별 전수 채점 + **심박수 정합** 채점
#
#   점수는 Q7-H 의 두 템플릿 `‖b−medN‖ − ‖b−medS‖` 를 **그대로** 쓴다(교차적합 · R22).
#   바꾸는 건 **어느 구간을 보느냐** 하나뿐이다.
from sklearn.metrics import roc_auc_score

def dist(B, ref):
    d = B - ref[None]
    return np.sqrt((d * d).sum(axis=(1, 2)))

def two_template_cv(B, tt, K, seed, n_rep):
    """되풀이 K겹 교차적합 두 템플릿 점수 (R22). 반환 (점수평균, 되풀이간 AUROC SD)."""
    accs, aus = [], []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any():
                continue
            if int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None, None
            medN = np.median(B[tr & ~tt], axis=0); medS = np.median(B[tr & tt], axis=0)
            sc[te] = dist(B[te], medN) - dist(B[te], medS)
        accs.append(sc); aus.append(roc_auc_score(tt.astype(int), sc))
    return np.mean(np.stack(accs), axis=0), (float(np.std(aus, ddof=1)) if len(aus) > 1 else 0.0)

def boot_auroc(tt, sc, seed, nb):
    rng = np.random.RandomState(seed); v = []
    for _ in range(nb):
        j = rng.randint(0, len(tt), len(tt)); tj = tt[j]
        if 0 < tj.sum() < len(tj):
            v.append(roc_auc_score(tj.astype(int), sc[j]))
    if len(v) < 20:
        return float("nan")
    return float(np.std(v, ddof=1))

def matched_auc(sc, tt, pre_v, band_w, min_s, min_n):
    """★ **같은 pre_rr 대역 안에서만** S vs N 쌍을 세는 조건부 일치도.

    템플릿은 **다시 적합하지 않는다** — 평가만 제한한다(한 번에 하나만 바꾼다).
    반환 (조건부 AUROC, 남은 S, 남은 N, 쌍 수, 쓸모 있던 대역 수)."""
    b = np.floor(pre_v / max(band_w, 1e-9)).astype(np.int64)
    num = den = 0.0; ks = kn = nb_ = 0
    for bb in np.unique(b):
        m = b == bb
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if len(s_) < min_s or len(n_) < min_n:
            continue
        ks += len(s_); kn += len(n_); nb_ += 1
        gt = float((s_[:, None] > n_[None, :]).sum())
        eq = float((s_[:, None] == n_[None, :]).sum())
        num += gt + 0.5 * eq; den += float(len(s_) * len(n_))
    if den < 1:
        return float("nan"), ks, kn, 0.0, nb_
    return num / den, ks, kn, den, nb_

run.log("\n" + "=" * 100)
run.log(f"【F-B】 구간별 전수 채점 ({K_FOLD}겹 교차적합 × {N_REPEAT}회)")
run.log("=" * 100)
PER, SKIP = {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    ns, nn = int(tt.sum()), int((~tt).sum())
    if ns < MIN_S_TPL or nn < MIN_S_TPL:
        SKIP.append((int(r), f"S {ns} · N {nn} — 템플릿 최소 {MIN_S_TPL} 미달")); continue
    Bb, pre_m = BEAT[mm], PRE[mm]
    row = dict(n=int(len(mm)), pos=ns, prev=float(tt.mean()),
               rr_med_s=float(np.median(pre_m[tt])), rr_med_n=float(np.median(pre_m[~tt])))
    row["rr_gap"] = float((row["rr_med_n"] - row["rr_med_s"]) / max(np.median(pre_m), 1e-9))
    row["ov_gap_early"] = float(t_overlap(pre_m[tt], SEGS["P_early"]).mean()
                                - t_overlap(pre_m[~tt], SEGS["P_early"]).mean())
    row["ov_gap_late"] = float(t_overlap(pre_m[tt], SEGS["P_late"]).mean()
                               - t_overlap(pre_m[~tt], SEGS["P_late"]).mean())
    SC = {}
    bad = False
    for k_, (a_, b_) in SEGS.items():
        sc_, rsd = two_template_cv(Bb[:, :, a_:b_], tt, K_FOLD, SEED0, N_REPEAT)
        if sc_ is None:
            bad = True; break
        SC[k_] = sc_
        row[k_] = float(roc_auc_score(tt.astype(int), sc_))
        se_ = boot_auroc(tt, sc_, SEED0 + int(r), NB_REC)
        row[k_ + "_se"] = float(np.sqrt(se_ ** 2 + rsd ** 2)) if np.isfinite(se_) else float("nan")
    if bad:
        SKIP.append((int(r), f"{K_FOLD}겹 중 한 겹의 훈련쪽 클래스가 2개 미만")); continue
    row["rr"] = float(roc_auc_score(tt.astype(int), np.median(pre_m) - pre_m))
    # ── 심박수 정합 (F1) — P_full 과 P_late 에 건다
    bw = max(BAND_MIN, BAND_FRAC * np.median(pre_m))
    row["band_w"] = float(bw)
    for k_ in ("P_full", "P_late"):
        a_, ks, kn, pr, nb_ = matched_auc(SC[k_], tt, pre_m, bw, MIN_BAND_S, MIN_BAND_N)
        row[k_ + "_matched"] = a_
        if k_ == "P_full":
            row.update(match_s=int(ks), match_n=int(kn), match_pairs=float(pr),
                       match_bands=int(nb_), match_s_frac=float(ks / max(ns, 1)))
    row["match_ok"] = bool(row["match_s"] >= MIN_MATCH_S and row["match_pairs"] >= MIN_MATCH_PAIR)
    PER[int(r)] = row

run.log(f"  채점 {len(PER)}개체 · **제외 {len(SKIP)}개체** (조용히 빼지 않는다 — 전부 아래에)")
for r, why in SKIP:
    run.log(f"    #{r}  {why}")
if len(PER) < 10:
    raise AssetError("채점된 개체가 너무 적다 — 자산·문턱을 확인할 것")
RS = sorted(PER)
A = {k: np.array([PER[r].get(k, np.nan) for r in RS], float) for k in
     ("P_full", "P_early", "P_late", "STT", "rr", "P_full_matched", "P_late_matched",
      "prev", "pos", "rr_gap", "ov_gap_early", "ov_gap_late", "match_s_frac", "match_pairs")}
MOK = np.array([PER[r]["match_ok"] for r in RS], bool)

# ── ★ 정합의 대가를 **관문보다 먼저** 낸다 (R17 — 조용히 적은 n 으로 판정하지 않는다)
run.log(f"\n  ★ 심박수 정합의 대가 — 정합 가능 개체 **{int(MOK.sum())}/{len(RS)}**"
        f" (문턱: 남은 S ≥ {MIN_MATCH_S} · 쌍 ≥ {MIN_MATCH_PAIR})")
run.log(f"    남은 S 비율 중앙 {np.nanmedian(A['match_s_frac']):.3f}"
        f" · 쌍 수 중앙 {np.nanmedian(A['match_pairs']):,.0f}"
        f" · S/N 중앙 RR 격차 중앙 {np.nanmedian(A['rr_gap']):.3f}")
bad_m = [int(r) for r, ok in zip(RS, MOK) if not ok]
run.log(f"    정합 불가 개체 {len(bad_m)}개: {bad_m or '없음'}")

run.log(f"\n  {'구간':<24}{'매크로':>9}{'SD':>9}{'중앙':>9}{'최소':>9}{'<0.5':>7}")
NM = {"rr": "RR·위치형 (대조)", "P_full": "P_full  (Q7-H 재현)",
      "P_early": "P_early (직전 T 자리)", "P_late": "P_late  (P 파 자리)",
      "STT": "STT     (음성 대조)"}
for k in ("rr", "P_full", "P_early", "P_late", "STT"):
    v = A[k]; f = np.isfinite(v)
    run.log(f"  {NM[k]:<24}{np.nanmean(v):>9.4f}{np.nanstd(v[f], ddof=1):>9.4f}"
            f"{np.nanmedian(v):>9.4f}{np.nanmin(v):>9.4f}{int((v[f] < 0.5).sum()):>7}")
run.log(f"  {'P_full 정합':<24}{np.nanmean(A['P_full_matched'][MOK]):>9.4f}"
        f"   (정합 가능 {int(MOK.sum())}개체만)")
run.log("\n  ⚠️ 두 템플릿은 **라벨을 쓰는 상한**이다. 학습 모델 0.8842 와 비교 금지.")
CONFIG["per_record"] = {str(r): PER[r] for r in RS}
CONFIG["skipped"] = [{"rec": r, "why": w} for r, w in SKIP]
CONFIG["match_cost"] = dict(n_ok=int(MOK.sum()), n=len(RS), unusable=bad_m)
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【F-C】 관문 F1·F2·F3
def boot_diff(a, b, seed, nb=NB_BOOT, mask=None):
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    if mask is not None:
        d = d[mask]
    d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)])
    return float(d.mean()), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)), len(d)

run.log("\n" + "=" * 100)
run.log("【F-C】 관문 — 형태인가 밀린 창인가")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

# ── F1 심박수 정합 (개체 수가 모자라면 **미결** — 조용히 적은 n 으로 판정하지 않는다)
if int(MOK.sum()) < MIN_MATCH_REC:
    VERD["F1"] = "⚠️ 미결"
    DIFF["F1"] = dict(n=int(MOK.sum()), reason="정합 가능 개체 부족")
    run.log(f"  F1  ⚠️ 미결  정합 가능 개체 {int(MOK.sum())} < {MIN_MATCH_REC}"
            f" — S 와 N 의 RR 분포가 겹치지 않는다. **이것 자체가 결과다**")
else:
    m1, lo1, hi1, n1_ = boot_diff(A["P_full_matched"], A["P_full"], SEED0 + 1, mask=MOK)
    DIFF["F1"] = dict(mean=m1, lo=lo1, hi=hi1, n=n1_)
    g_("F1", decide(lo1, hi1, -NI_MATCH, ">"),
       f"짝지은 (정합 − 무정합) **{m1:+.4f}** [{lo1:+.4f}, {hi1:+.4f}] · n={n1_}"
       f" vs 여유 −{NI_MATCH}   [지지 = 형태가 진짜]")
    m1b, lo1b, hi1b, _ = boot_diff(A["P_late_matched"], A["P_late"], SEED0 + 2, mask=MOK)
    run.log(f"    (보조) P_late 정합 − 무정합 {m1b:+.4f} [{lo1b:+.4f}, {hi1b:+.4f}]")
    DIFF["F1_late"] = dict(mean=m1b, lo=lo1b, hi=hi1b)

# ── F2 신호의 자리
m2, lo2, hi2, n2_ = boot_diff(A["P_late"], A["P_early"], SEED0 + 3)
DIFF["F2"] = dict(mean=m2, lo=lo2, hi=hi2, n=n2_)
g_("F2", decide(lo2, hi2, 0.0, ">"),
   f"짝지은 (P_late − P_early) **{m2:+.4f}** [{lo2:+.4f}, {hi2:+.4f}] · n={n2_}"
   f" · late 가 나은 개체 {int(np.nansum(A['P_late'] > A['P_early']))}/{n2_}   [지지 = 형태가 진짜]")

# ── F3 음성 대조
m3, lo3, hi3, n3_ = boot_diff(A["P_full"], A["STT"], SEED0 + 4)
DIFF["F3"] = dict(mean=m3, lo=lo3, hi=hi3, n=n3_)
g_("F3", decide(lo3, hi3, F3_MARGIN, ">"),
   f"짝지은 (P_full − STT) **{m3:+.4f}** [{lo3:+.4f}, {hi3:+.4f}] vs 여유 {F3_MARGIN}"
   f"   [지지 = 신호가 P **창** 안에 있다 — **P 파라는 뜻은 아니다**]")
run.log(f"    STT 매크로 {np.nanmean(A['STT']):.4f} — 이게 P_full({np.nanmean(A['P_full']):.4f})"
        " 에 가까우면 「P 파를 읽는다」는 성립하지 않는다")

# ── 최대 기여 개체 (R11-c)
d2 = A["P_late"] - A["P_early"]; fin = np.isfinite(d2)
tot = float(np.abs(d2[fin]).sum())
top = sorted(np.where(fin)[0], key=lambda i: -abs(d2[i]))[:5]
run.log(f"\n  F2 기여 상위 5 (|Δ| 합 {tot:.3f} 중):")
for i in top:
    run.log(f"    #{RS[i]}  late−early {d2[i]:+.4f} ({abs(d2[i])/max(tot,1e-9)*100:.1f}%)"
            f" · 유병률 {A['prev'][i]:.3f} · T중첩격차 {A['ov_gap_early'][i]:+.3f}"
            f" · RR격차 {A['rr_gap'][i]:+.3f}")
if top:
    m2b, lo2b, hi2b, _ = boot_diff(np.delete(A["P_late"], top[0]),
                                   np.delete(A["P_early"], top[0]), SEED0 + 5)
    run.log(f"  최대 기여 개체 #{RS[top[0]]} 제외 → **{m2b:+.4f}** [{lo2b:+.4f}, {hi2b:+.4f}]")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【F-D】 관문 F4 ⚠️ **지지 = 교란이 있다**
def boot_rho(x, y, seed, nb=NB_BOOT):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 5:
        return float("nan"), float("nan"), float("nan"), 0
    r0 = float(stats.spearmanr(x, y).statistic)
    rng = np.random.RandomState(seed); v = []
    for _ in range(nb // 4):
        j = rng.randint(0, len(x), len(x))
        if len(np.unique(x[j])) > 2 and len(np.unique(y[j])) > 2:
            v.append(stats.spearmanr(x[j], y[j]).statistic)
    if len(v) < 20:
        return r0, float("nan"), float("nan"), len(x)
    return r0, float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)), len(x)

run.log("\n" + "=" * 100)
run.log("【F-D】 관문 F4 — ⚠️ **지지 = 교란이 있다** (다른 관문과 방향이 반대)")
run.log("=" * 100)
r4, l4, h4, n4 = boot_rho(A["ov_gap_early"], A["P_early"], SEED0 + 11)
DIFF["F4"] = dict(rho=r4, lo=l4, hi=h4, n=n4)
g_("F4", decide(l4, h4, 0.0, ">"),
   f"rho(직전T 중첩격차, P_early AUROC) **{r4:+.3f}** [{l4:+.3f}, {h4:+.3f}] · n={n4}"
   f"   ⚠️ **지지 = 교란 있음**")
r4b, l4b, h4b, _ = boot_rho(A["ov_gap_early"], A["P_late"], SEED0 + 12)
run.log(f"    (대조) rho(중첩격차, **P_late**) {r4b:+.3f} [{l4b:+.3f}, {h4b:+.3f}]"
        "  ← P_late 은 T 가 안 드는 자리라 여기선 상관이 약해야 한다")
r4c, l4c, h4c, _ = boot_rho(A["rr_gap"], A["P_early"], SEED0 + 13)
run.log(f"    (보조) rho(S/N RR 격차, P_early) {r4c:+.3f} [{l4c:+.3f}, {h4c:+.3f}]")

run.log("\n  T중첩격차 층별")
for lo_, hi_ in ((-9, 0.02), (0.02, 0.10), (0.10, 0.25), (0.25, 9)):
    s = (A["ov_gap_early"] >= lo_) & (A["ov_gap_early"] < hi_)
    if s.sum():
        run.log(f"    [{lo_:.2f},{hi_:.2f}) {int(s.sum()):>2}개체 · 유병률 {np.nanmean(A['prev'][s]):.3f}"
                f" · P_early {np.nanmean(A['P_early'][s]):.4f}"
                f" · P_late {np.nanmean(A['P_late'][s]):.4f}"
                f" · P_full {np.nanmean(A['P_full'][s]):.4f}")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
run.save_json("config", CONFIG)

# ── 결론은 **관문 조합으로만** 만든다
ok = lambda k: VERD.get(k, "").startswith("✅")
no = lambda k: VERD.get(k, "").startswith("❌")
run.log("\n" + "=" * 100)
if ok("F1") and ok("F2") and ok("F3") and not ok("F4"):
    run.log("  ★ **형태 축은 진짜다.** Q7-D·E·H 의 해석을 유지하고 Q7-G(교차환자 템플릿)로 간다")
elif (no("F1") or no("F2")) and ok("F4"):
    run.log("  ⛔ **리듬 대리였다.** Q7-D·E·H 의 「형태 축」 해석을 **소급 정정**하고,")
    run.log("     Q7-G 는 취소한다 — 전이시킬 형태 신호가 애초에 없다. 큐는 Q7-I 로 직행")
elif not ok("F3"):
    run.log("  ⛔ **신호가 P 창에 있지도 않다** — ST/T 구간이 P 구간만큼 맞힌다.")
    run.log("     읽고 있는 게 P 창 내용물이라는 주장조차 못 한다")
else:
    run.log("  ⚠️ **갈렸다.** 억지로 한쪽으로 읽지 않는다 — 관문표를 그대로 보고한다(R18)")
    run.log("     갈린 조합 자체가 다음 실험의 사전등록 재료다")
run.log("=" * 100)

In [ ]:
# CELL 7 — 【F-E】 리듬·기호·런 구조 층화 — **관문 아님 · 진단**
#
#   ★ `svdb_data5.npz` 의 `rhythm`·`rhythm_names`·`sym` 은 Q7-D 부터 **한 번도 안 읽었다.**
#     여기서 처음 읽는다. 목적은 「이 코호트에서 우리 가정이 어디서 깨지나」를 **보이게**
#     만드는 것이고, **관문을 다시 매기지 않는다.**
run.log("\n" + "=" * 100)
run.log("【F-E】 리듬 · 기호 · 런 구조 — 관문 아님(진단)")
run.log("=" * 100)

# ── 자산 확인. **없으면 추측하지 않고 생략한다** (R16)
RHY = RHN = SYM = None
_f = set(getattr(d5, "files", []) or [])
if {"rhythm", "rhythm_names"} <= _f:
    _r = np.asarray(d5["rhythm"]); _n = [str(x) for x in np.asarray(d5["rhythm_names"]).tolist()]
    if len(_r) == len(keep):
        RHY = _r[keep]; RHN = _n
    else:
        run.log(f"  ⚠️ rhythm 길이 {len(_r)} ≠ keep {len(keep)} — 리듬 층화 **생략**(맞춰 넣지 않는다)")
else:
    run.log("  ⚠️ npz 에 rhythm/rhythm_names 가 없다 — 리듬 층화 **생략**")
if "sym" in _f:
    _s = np.asarray(d5["sym"])
    if len(_s) == len(keep):
        SYM = np.array([str(x) for x in _s[keep]])
    else:
        run.log(f"  ⚠️ sym 길이 {len(_s)} ≠ keep {len(keep)} — 기호 분해 **생략**")
else:
    run.log("  ⚠️ npz 에 sym 이 없다 — 기호 분해 **생략**")

DIAG = {}
_tt_all = (Y == IDX_S)

# ── ① 리듬 라벨 분포 + **AF 구간 안 비트가 실제로 무슨 기호인지**
if RHY is not None:
    from collections import Counter
    cnt = Counter(RHN[i] for i in RHY)
    run.log(f"\n  ① 리듬 라벨 {len(RHN)}종 · 비트 분포 상위 8: "
            + ", ".join(f"{k} {v:,}" for k, v in cnt.most_common(8)))
    AFLIKE = [n for n in RHN if n.upper() in ("AFIB", "AFL", "AF", "SVTA", "SBR", "B", "T")]
    run.log(f"     이 퀘스트 가정을 깨는 리듬: {AFLIKE or '없음'}")
    for nm_ in AFLIKE:
        k_ = RHN.index(nm_); m_ = RHY == k_
        if not m_.any():
            continue
        line = f"     {nm_:<6} 비트 {int(m_.sum()):>7,}"
        if SYM is not None:
            sc = Counter(SYM[m_]).most_common(5)
            line += "  기호 " + ", ".join(f"{a}:{b:,}" for a, b in sc)
            n_frac = float((Y[m_] == 0).mean())
            line += f"   → AAMI **N 비율 {n_frac:.3f}**"
        recs_ = sorted({int(r) for r in REC[m_]})
        line += f"  ({len(recs_)}개체)"
        run.log(line)
    run.log("     ★ AF/AFL 의 AAMI N 비율이 1 에 가까우면 **AF 는 이 라벨 체계에서 N 이다** —")
    run.log("       「AF 를 맞히나」를 이 데이터로는 물을 수 없다는 뜻이고, 규약 예상의 실측 확인이다")
    DIAG["rhythm_counts"] = {k: int(v) for k, v in cnt.items()}

# ── ② S 하위기호 분해 — `J`(접합부)는 역행성 P 라 **STT 구간**에 신호가 있다
if SYM is not None:
    from collections import Counter
    sub = Counter(SYM[_tt_all])
    run.log(f"\n  ② S 클래스 하위기호: " + ", ".join(f"{k} {v:,}" for k, v in sub.most_common()))
    jr = {}
    for r in RS:
        m_ = (REC == r) & _tt_all
        if m_.sum():
            jr[int(r)] = float(np.mean(np.isin(SYM[m_], ["J"])))
    hot = sorted((v, k) for k, v in jr.items() if v > 0.2)[::-1]
    run.log(f"     `J`(접합부) 비율 > 0.2 인 개체 {len(hot)}개: "
            + (", ".join(f"#{k}({v:.2f})" for v, k in hot[:8]) or "없음"))
    run.log("     ★ `J` 는 역행성 P 라 신호가 **QRS/초기 ST** 에 있다 —")
    run.log("       그 개체에서는 **F3 의 음성 대조(STT) 가정이 깨진다**. F3 을 그대로 읽지 않는다")
    DIAG["s_subsymbols"] = {k: int(v) for k, v in sub.items()}
    DIAG["j_heavy"] = [int(k) for _, k in hot]

# ── ③ 런 구조 — **고립 S** 만이 「기저보다 이르다」가 성립하는 비트다
run.log("\n  ③ 런 구조 (레코드 내 시간순 · 고립 = 앞뒤가 모두 비-S)")
ISO = {}
for r in RS:
    mm = np.where(REC == r)[0]              # 레코드 안에서는 시간순으로 저장돼 있다
    t_ = (Y[mm] == IDX_S)
    if not t_.any():
        continue
    prev_ = np.r_[False, t_[:-1]]; next_ = np.r_[t_[1:], False]
    iso = t_ & ~prev_ & ~next_
    # 최장 런
    run_len = mx = 0
    for v in t_:
        run_len = run_len + 1 if v else 0
        mx = max(mx, run_len)
    ISO[int(r)] = dict(iso_frac=float(iso.sum() / max(t_.sum(), 1)), max_run=int(mx),
                       n_iso=int(iso.sum()))
IF = np.array([ISO[r]["iso_frac"] for r in RS if r in ISO])
MR = np.array([ISO[r]["max_run"] for r in RS if r in ISO])
run.log(f"     고립 S 비율 — 중앙 {np.median(IF):.3f} · 최소 {IF.min():.3f} · 최대 {IF.max():.3f}")
run.log(f"     최장 런 — 중앙 {np.median(MR):.0f} · 최대 {MR.max()}")
runny = sorted(((ISO[r]["iso_frac"], r) for r in RS if r in ISO))[:8]
run.log("     고립 비율이 가장 낮은(=런이 지배적인) 8개체: "
        + ", ".join(f"#{r}(고립 {v:.2f} · 최장런 {ISO[r]['max_run']})" for v, r in runny))
run.log("     ★ 런 안의 2번째 이후 비트는 **직전 비트 대비 이르지 않다.** 레코드 중앙 RR")
run.log("       기준이라 「이르다」로 찍힐 뿐이다 — SVT·심방빈맥이 이 모양이다")
DIAG["run_structure"] = {str(k): v for k, v in ISO.items()}

# ── ④ 층별로 관문 값을 다시 낸다 (판정이 아니라 **분해**)
def _strat(name, mask):
    if mask.sum() < 3:
        return
    run.log(f"     {name:<22}{int(mask.sum()):>3}개체 · P_early {np.nanmean(A['P_early'][mask]):.4f}"
            f" · P_late {np.nanmean(A['P_late'][mask]):.4f}"
            f" · P_full {np.nanmean(A['P_full'][mask]):.4f}"
            f" · RR {np.nanmean(A['rr'][mask]):.4f}")
run.log("\n  ④ 층별 분해 (관문 재판정 아님)")
ifa = np.array([ISO.get(r, {}).get("iso_frac", np.nan) for r in RS])
_strat("고립 S 우세 (≥0.7)", ifa >= 0.7)
_strat("혼합 (0.3~0.7)", (ifa >= 0.3) & (ifa < 0.7))
_strat("런 우세 (<0.3)", ifa < 0.3)
if SYM is not None:
    jv = np.array([jr.get(r, np.nan) for r in RS])
    _strat("J 비율 > 0.2", jv > 0.2)
    _strat("J 비율 ≤ 0.2", jv <= 0.2)
run.log("\n  ⚠️ **여기서 관문을 다시 매기지 않는다.** 이 층화는 Q7-I 의 사전등록 재료다 —")
run.log("     리듬 특징을 넣을 때 **고립 S / 런 / AF 개체를 따로 보고**한다")
CONFIG["diagnostics"] = DIAG
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 그림 + 마무리
#  ★ Colab 기본 폰트에 한글이 없어 □ 로 깨진다. 그림 라벨은 ASCII 로 쓴다.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
rs_ = np.array(RS)

ax[0].scatter(A["P_early"], A["P_late"], s=34, alpha=.85,
              c=A["ov_gap_early"], cmap="magma_r")
ax[0].plot([0, 1], [0, 1], color="gray", ls="--", lw=1)
ax[0].axhline(.5, color="crimson", ls=":", lw=1); ax[0].axvline(.5, color="crimson", ls=":", lw=1)
cb = plt.colorbar(ax[0].collections[0], ax=ax[0]); cb.set_label("prev-T overlap gap (S-N)", fontsize=8)
ax[0].set_xlabel("AUROC, P_early  (idx 0-32, where prev T lands)")
ax[0].set_ylabel("AUROC, P_late  (idx 53-85, where P lives)")
ax[0].set_title(f"(1) where is the signal?  (n={len(rs_)})")
ax[0].grid(alpha=.3); ax[0].set_xlim(-.02, 1.02); ax[0].set_ylim(-.02, 1.02)

KS = ["rr", "P_full", "P_early", "P_late", "STT"]
LB = ["RR positional\n(control)", "P_full\n(Q7-H window)", "P_early\n(prev-T zone)",
      "P_late\n(P-wave zone)", "STT\n(negative control)"]
CL = ["C7", "C0", "C3", "C2", "C1"]
ax[1].bar(range(len(KS)), [float(np.nanmean(A[k])) for k in KS], color=CL)
if int(MOK.sum()) >= 3:
    ax[1].scatter([1], [float(np.nanmean(A["P_full_matched"][MOK]))], marker="_", s=900,
                  color="k", zorder=5, label="P_full, rate-matched")
    ax[1].legend(fontsize=8)
ax[1].axhline(0.5, color="crimson", ls="--", lw=1)
ax[1].set_xticks(range(len(KS))); ax[1].set_xticklabels(LB, fontsize=7)
ax[1].set_ylabel("macro AUROC (two-template, ORACLE)")
ax[1].set_title("(2) macro by window   F3 = P_full vs STT (window, not P-wave)")
ax[1].grid(alpha=.3, axis="y"); ax[1].set_ylim(0, 1.05)

ax[2].scatter(A["ov_gap_early"], A["P_early"], s=34, alpha=.85, label="P_early")
ax[2].scatter(A["ov_gap_early"], A["P_late"], s=34, alpha=.85, marker="^", label="P_late")
ax[2].axhline(.5, color="crimson", ls="--", lw=1)
ax[2].set_xlabel("prev-T overlap gap in P_early  (S - N, modelled)")
ax[2].set_ylabel("AUROC")
ax[2].set_title(f"(3) F4 confound check   rho={DIFF['F4']['rho']:+.3f} (support = CONFOUNDED)")
ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
plt.tight_layout(); run.save_fig("q7f_window_confound", fig); plt.show()

run.log("\n" + "=" * 100)
run.log("관문 요약  (F4 만 지지의 방향이 반대 — 지지 = 교란 있음)")
run.log("=" * 100)
for k in ("F1", "F2", "F3", "F4"):
    run.log(f"  {k:<4}{VERD.get(k, '(미실행)')}")
run.finish({"verdicts": VERD, "diffs": DIFF,
            "macro": {k: float(np.nanmean(A[k])) for k in KS},
            "macro_matched": float(np.nanmean(A["P_full_matched"][MOK])) if MOK.any() else None,
            "n_scored": len(RS), "n_skipped": len(SKIP), "n_match_ok": int(MOK.sum())})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-window-confound`")